# AI623 PA2 — LLM Alignment
**SFT · Reward Model · DPO · PPO · GRPO · RLVR**

Run this notebook top-to-bottom on a Colab T4 GPU.
Toggle the boolean flags in the **Config** section to choose which methods to run.

> **Constraint reminder:** No TRL / trlX / OpenRLHF trainers allowed. All alignment logic is implemented from scratch in `alignment/`.

## 1 · Runtime Sanity Checks

In [ ]:
import sys, torch
print('Python :', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f'GPU    : {props.name}  VRAM={vram_gb:.1f} GB')
    if vram_gb < 12:
        print('WARNING: < 12 GB VRAM — PPO may OOM. Reduce prompts_per_step or max_new_tokens.')
    else:
        print('OK: sufficient VRAM for all methods')
else:
    print('ERROR: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')


## 2 · Clone / Mount Repo

Set `REPO_URL` to your GitHub URL, or set `USE_LOCAL_DIR = True` if the notebook is already inside the repo directory.

In [ ]:
import os
from pathlib import Path

REPO_URL       = 'https://github.com/YOUR_USERNAME/dvlm_pa2.git'  # ← edit this
USE_LOCAL_DIR  = False   # True if running from inside the repo already

if not USE_LOCAL_DIR:
    if not Path('dvlm_pa2').exists():
        !git clone {REPO_URL}
    %cd dvlm_pa2
else:
    # Already inside repo root — just confirm
    assert Path('train_sft.py').exists(), 'Not in repo root — check your working directory'

print('Working dir:', os.getcwd())
print('Repo files :', sorted([p.name for p in Path('.').iterdir() if p.is_file()]))


## 3 · Install Dependencies

In [ ]:
!pip install -r requirements.txt -q
print('Dependencies installed.')


## 4 · Hugging Face Login

Required for `meta-llama/Llama-3.2-1B-Instruct`.

**Option A (recommended for Colab):** Add `HF_TOKEN` to Colab Secrets (key icon in left sidebar).

**Option B:** Paste your token directly below (do NOT commit it to git).

If you haven't accepted the Llama-3.2 license: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

In [ ]:
import os
from huggingface_hub import login

# Try Colab secrets first, then env var, then prompt
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets.')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('HF_TOKEN loaded from environment.')

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in to Hugging Face.')
else:
    print('WARNING: HF_TOKEN not found.')
    print('Reward model and value model require Llama-3.2 access.')
    print('Set HF_TOKEN in Colab Secrets or run: login(token="hf_...")')


## 5 · Autoreload (for iterative dev)

In [ ]:
%load_ext autoreload
%autoreload 2


## 6 · Imports

In [ ]:
import sys, torch
from pathlib import Path

# Ensure repo root is on path
repo_root = Path('.').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Core utils
from utils.config   import load_config, merge_configs
from utils.seed     import set_seed
from utils.memory   import print_memory, clear_cache, memory_stats
from utils.logging_utils import get_logger, MetricsLogger
from utils.io       import ensure_dir, save_peft_checkpoint
from utils.plotting import plot_training_curves, plot_reward_distribution, plot_method_comparison
from utils.metrics  import preference_accuracy, win_rate

# Data
from data.hh_rlhf  import load_hh_rlhf, SFTDataset, RMDataset, DPODataset, PromptDataset
from data.gsm8k    import load_gsm8k, GSM8KDataset
from data.collators import SFTCollator, RMCollator, DPOCollator
from data.parsing  import parse_hh_example, extract_gsm8k_answer, answers_match

# Model
from model.loading      import load_policy, load_reward_backbone, get_tokenizer
from model.lora         import apply_lora, freeze_model
from model.reward_model import RewardModel, score_texts
from model.value_model  import ValueModel
from model.generation   import generate_responses, generate_k_responses
from model.logprobs     import token_logprobs, sequence_logprobs

# Alignment
from alignment.dpo       import dpo_loss, dpo_step
from alignment.ppo       import ppo_rollout, ppo_update, ppo_sanity_checks
from alignment.grpo      import grpo_rollout, grpo_update
from alignment.rlvr      import rlvr_rollout, eval_gsm8k_pass_at_1
from alignment.kl        import kl_from_ref
from alignment.advantages import compute_gae_advantages, normalise_advantages

print('All imports OK.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)


## 7 · Config

Set the flags below to control which training stages run.

In [ ]:
# ── Run flags ──────────────────────────────────────────────────────────────
RUN_SFT   = True
RUN_RM    = True
RUN_DPO   = True
RUN_PPO   = True
RUN_GRPO  = True
RUN_RLVR  = True
RUN_EVAL  = True

# ── Dataset size (reduce for quick smoke-test) ──────────────────────────────
MAX_TRAIN_SAMPLES = 3000   # None = full dataset
MAX_TEST_SAMPLES  = 500

# ── Paths ───────────────────────────────────────────────────────────────────
SFT_CKPT  = 'runs/sft/sft_adapter'
RM_CKPT   = 'runs/rm/rm.pt'
RUNS_DIR  = 'runs'

# ── Load configs ────────────────────────────────────────────────────────────
default_cfg = load_config('configs/default.yaml')
sft_cfg     = merge_configs(default_cfg, load_config('configs/sft.yaml'))
rm_cfg      = merge_configs(default_cfg, load_config('configs/rm.yaml'))
dpo_cfg     = merge_configs(default_cfg, load_config('configs/dpo.yaml'))
ppo_cfg     = merge_configs(default_cfg, load_config('configs/ppo.yaml'))
grpo_cfg    = merge_configs(default_cfg, load_config('configs/grpo.yaml'))
rlvr_cfg    = merge_configs(default_cfg, load_config('configs/rlvr.yaml'))

set_seed(default_cfg.get('seed', 42))
print('Config loaded. Run flags:')
for flag, val in [('SFT',RUN_SFT),('RM',RUN_RM),('DPO',RUN_DPO),
                  ('PPO',RUN_PPO),('GRPO',RUN_GRPO),('RLVR',RUN_RLVR),('EVAL',RUN_EVAL)]:
    print(f'  {flag}: {val}')


## 8 · Data Sanity Checks

Load 3 parsed HH-RLHF examples and verify prompt/response splitting.

In [ ]:
print('Loading HH-RLHF (train) ...')
hh_train = load_hh_rlhf('train', max_samples=MAX_TRAIN_SAMPLES)
hh_test  = load_hh_rlhf('test',  max_samples=MAX_TEST_SAMPLES)

print('\n--- 3 parsed HH-RLHF examples ---')
for i, ex in enumerate(hh_train[:3]):
    print(f'\n[{i}] PROMPT  (last 200 chars): ...{ex["prompt"][-200:]}')
    print(f'    CHOSEN  (first 150 chars): {ex["chosen"][:150]}...')
    print(f'    REJECTED(first 150 chars): {ex["rejected"][:150]}...')

assert all('prompt' in ex and 'chosen' in ex and 'rejected' in ex for ex in hh_train[:10])
assert all(len(ex['prompt']) > 0 and len(ex['chosen']) > 0 for ex in hh_train[:10])
print('\nData sanity checks PASSED.')


In [ ]:
print('Loading GSM8K ...')
gsm_train = load_gsm8k('train', max_samples=500)
gsm_test  = load_gsm8k('test',  max_samples=200)

print('\n--- 3 GSM8K examples ---')
for ex in gsm_train[:3]:
    print(f'Q: {ex["question"][:100]}...')
    print(f'Gold answer: {ex["gold_answer"]}')

# Verify extractor on gold solutions
from datasets import load_dataset
raw_gsm = load_dataset('openai/gsm8k', 'main', split='train')
correct = sum(
    answers_match(extract_gsm8k_answer(ex['answer']), extract_gsm8k_answer(ex['answer']))
    for ex in list(raw_gsm)[:20]
)
print(f'\nExtractor self-check on 20 gold solutions: {correct}/20 correct')
assert correct == 20, 'Extractor failed on gold solutions!'
print('GSM8K sanity checks PASSED.')


## 9 · Model Loading Sanity Checks

Load policy, reference, RM, and value model backbones and print parameter counts + VRAM.

In [ ]:
import torch

print('Loading policy tokenizer ...')
policy_tok = get_tokenizer(default_cfg['policy_model'], padding_side='left')
print(f'  pad_token={policy_tok.pad_token!r}  padding_side={policy_tok.padding_side}')

print('\nLoading policy model ...')
policy = load_policy(
    default_cfg['policy_model'],
    dtype=default_cfg.get('dtype', 'bfloat16'),
    gradient_checkpointing=True,
    device_map=DEVICE,
)
policy = apply_lora(policy,
    r=default_cfg.get('lora_r', 8),
    alpha=default_cfg.get('lora_alpha', 16),
    dropout=default_cfg.get('lora_dropout', 0.05),
    target_modules=default_cfg.get('lora_target_modules', ['q_proj', 'v_proj']),
)
print_memory()

print('\nLoading reference model (frozen copy of policy) ...')
ref_model = load_policy(
    default_cfg['policy_model'],
    dtype=default_cfg.get('dtype', 'bfloat16'),
    gradient_checkpointing=False,
    device_map=DEVICE,
)
freeze_model(ref_model)
print_memory()

print('\nLoading RM backbone (Llama-3.2-1B) ...')
try:
    rm = RewardModel(default_cfg['reward_model'],
                     use_8bit=default_cfg.get('use_8bit_frozen', True))
    print_memory()
    print('RM loaded OK.')
except Exception as e:
    print(f'RM load failed: {e}')
    print('Ensure HF_TOKEN is set and you have accepted the Llama-3.2 license.')
    rm = None

print('\nModel loading complete.')


## 10 · SFT Warm-Up (C2)

Fine-tune the policy on HH-RLHF chosen responses. Response tokens only — prompt tokens are masked.
Saves checkpoint to `runs/sft/sft_adapter`.

In [ ]:
if RUN_SFT:
    import math
    from torch.utils.data import DataLoader
    from transformers import get_linear_schedule_with_warmup
    from tqdm.notebook import tqdm

    ensure_dir('runs/sft')
    mlog = MetricsLogger('runs/sft/metrics.jsonl')

    train_ds = SFTDataset(hh_train, policy_tok, sft_cfg.get('max_seq_len', 1024))
    collator = SFTCollator(policy_tok, sft_cfg.get('max_seq_len', 1024))
    loader   = DataLoader(train_ds, batch_size=sft_cfg.get('batch_size', 8),
                          shuffle=True, collate_fn=collator)

    optimizer   = torch.optim.AdamW(
        [p for p in policy.parameters() if p.requires_grad], lr=sft_cfg.get('lr', 2e-4)
    )
    grad_accum  = sft_cfg.get('grad_accum', 4)
    total_steps = math.ceil(len(loader) / grad_accum)
    scheduler   = get_linear_schedule_with_warmup(optimizer, 50, total_steps)

    policy.train()
    optimizer.zero_grad()
    running_loss, global_step = 0.0, 0

    for batch_idx, batch in enumerate(tqdm(loader, desc='SFT')):
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['labels'].to(DEVICE)
        loss  = policy(input_ids=ids, attention_mask=mask, labels=lbls).loss / grad_accum
        loss.backward()
        running_loss += loss.item() * grad_accum

        if (batch_idx + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            global_step += 1
            if global_step % 20 == 0:
                avg = running_loss / 20
                mlog.log(global_step, {'loss': avg, 'ppl': math.exp(min(avg, 20))})
                running_loss = 0.0

    save_peft_checkpoint(policy, SFT_CKPT)
    mlog.close()
    print(f'SFT complete. Checkpoint → {SFT_CKPT}')
    print_memory()
else:
    print('RUN_SFT=False — skipping.')


## 11 · Reward Model Training (C1)

Fine-tune `meta-llama/Llama-3.2-1B-Instruct` as a sequence classifier on HH-RLHF preference pairs.
Target: ≥60% preference accuracy on held-out test set.

In [ ]:
if RUN_RM:
    import torch.nn.functional as F
    from transformers import AutoTokenizer, get_linear_schedule_with_warmup
    from torch.utils.data import DataLoader
    from tqdm.notebook import tqdm

    if rm is None:
        print('ERROR: RM backbone failed to load. Cannot train RM.')
    else:
        ensure_dir('runs/rm')
        rm_tok = AutoTokenizer.from_pretrained(rm_cfg['reward_model'])
        rm_tok.padding_side = 'right'
        if rm_tok.pad_token is None:
            rm_tok.pad_token = rm_tok.eos_token

        collator = RMCollator(rm_tok, rm_cfg.get('max_seq_len', 1024))
        loader   = DataLoader(RMDataset(hh_train), batch_size=rm_cfg.get('batch_size', 8),
                              shuffle=True, collate_fn=collator)

        rm_opt = torch.optim.AdamW(
            [p for p in rm.parameters() if p.requires_grad], lr=rm_cfg.get('lr', 1e-4)
        )
        rm_sched = get_linear_schedule_with_warmup(rm_opt, 30, len(loader))
        mlog_rm  = MetricsLogger('runs/rm/metrics.jsonl')
        lambda_r = rm_cfg.get('lambda_reg', 1e-3)

        rm.train()
        for step, batch in enumerate(tqdm(loader, desc='RM')):
            r_pos = rm(batch['input_ids_pos'].to(DEVICE), batch['attention_mask_pos'].to(DEVICE))
            r_neg = rm(batch['input_ids_neg'].to(DEVICE), batch['attention_mask_neg'].to(DEVICE))
            loss  = -F.logsigmoid(r_pos - r_neg).mean() + lambda_r*(r_pos**2 + r_neg**2).mean()
            rm_opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0)
            rm_opt.step(); rm_sched.step()
            if step % 20 == 0:
                acc = (r_pos > r_neg).float().mean().item()
                mlog_rm.log(step, {'loss': loss.item(), 'pref_acc': acc})

        # Freeze after training
        rm.freeze()
        from utils.io import save_checkpoint
        save_checkpoint(rm, RM_CKPT)
        mlog_rm.close()
        print(f'RM training complete. Saved → {RM_CKPT}')
else:
    print('RUN_RM=False — skipping.')


## 12 · DPO Training (C4)

Direct Preference Optimisation — offline, no RM needed during training.
Sanity check: preference accuracy should start near 50% and rise above 55–60%.

In [ ]:
if RUN_DPO:
    from torch.utils.data import DataLoader
    from transformers import get_linear_schedule_with_warmup
    from tqdm.notebook import tqdm
    import math

    # Reload policy from SFT checkpoint for clean initialisation
    clear_cache()
    dpo_policy = load_policy(default_cfg['policy_model'],
                              dtype=default_cfg.get('dtype','bfloat16'),
                              gradient_checkpointing=True, device_map=DEVICE)
    dpo_policy = apply_lora(dpo_policy,
        r=default_cfg['lora_r'], alpha=default_cfg['lora_alpha'],
        dropout=default_cfg['lora_dropout'],
        target_modules=default_cfg['lora_target_modules'])
    if Path(SFT_CKPT).exists():
        dpo_policy.load_adapter(SFT_CKPT, adapter_name='default')
        print('Loaded SFT adapter for DPO policy.')

    dpo_ref = load_policy(default_cfg['policy_model'],
                           dtype=default_cfg.get('dtype','bfloat16'),
                           gradient_checkpointing=False, device_map=DEVICE)
    dpo_ref = apply_lora(dpo_ref, r=default_cfg['lora_r'], alpha=default_cfg['lora_alpha'],
                          dropout=0.0, target_modules=default_cfg['lora_target_modules'])
    if Path(SFT_CKPT).exists():
        dpo_ref.load_adapter(SFT_CKPT, adapter_name='default')
    freeze_model(dpo_ref)

    collator = DPOCollator(policy_tok, dpo_cfg.get('max_seq_len', 1024))
    loader   = DataLoader(DPODataset(hh_train), batch_size=dpo_cfg.get('batch_size', 8),
                          shuffle=True, collate_fn=collator)

    optimizer   = torch.optim.AdamW(
        [p for p in dpo_policy.parameters() if p.requires_grad],
        lr=dpo_cfg.get('lr', 5e-5))
    grad_accum  = dpo_cfg.get('grad_accum', 4)
    total_steps = math.ceil(len(loader) / grad_accum)
    scheduler   = get_linear_schedule_with_warmup(optimizer, 30, total_steps)
    mlog_dpo    = MetricsLogger('runs/dpo/metrics.jsonl')
    ensure_dir('runs/dpo')

    dpo_policy.train()
    optimizer.zero_grad()
    global_step = 0

    # Sanity: pref_acc should start ~0.5
    first_batch = next(iter(loader))
    with torch.no_grad():
        _, init_info = dpo_loss(dpo_policy, dpo_ref,
            first_batch['input_ids_pos'].to(DEVICE), first_batch['attention_mask_pos'].to(DEVICE),
            first_batch['input_ids_neg'].to(DEVICE), first_batch['attention_mask_neg'].to(DEVICE),
            first_batch['prompt_lens'].to(DEVICE), beta=dpo_cfg.get('beta', 0.1))
    print(f'DPO init pref_acc={init_info["pref_acc"]:.3f}  (expect ~0.50)')

    for batch_idx, batch in enumerate(tqdm(loader, desc='DPO')):
        info = dpo_step(dpo_policy, dpo_ref, batch, optimizer,
                        beta=dpo_cfg.get('beta', 0.1))
        if (batch_idx + 1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(dpo_policy.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            global_step += 1
            if global_step % 10 == 0:
                mlog_dpo.log(global_step, info)

    save_peft_checkpoint(dpo_policy, 'runs/dpo/adapter')
    mlog_dpo.close()
    print('DPO complete.')
    print_memory()
else:
    print('RUN_DPO=False — skipping.')


## 13 · PPO Training (C3)

Most memory-intensive method. Requires RM + value model.
Runs sanity checks (GAE, ratio, clipping) before training.

In [ ]:
if RUN_PPO:
    import random
    from tqdm.notebook import tqdm

    ppo_sanity_checks()

    clear_cache()
    ppo_policy = load_policy(default_cfg['policy_model'],
                              dtype=default_cfg.get('dtype','bfloat16'),
                              gradient_checkpointing=True, device_map=DEVICE)
    ppo_policy = apply_lora(ppo_policy,
        r=default_cfg['lora_r'], alpha=default_cfg['lora_alpha'],
        dropout=default_cfg['lora_dropout'],
        target_modules=default_cfg['lora_target_modules'])
    if Path(SFT_CKPT).exists():
        ppo_policy.load_adapter(SFT_CKPT, adapter_name='default')

    ppo_ref = load_policy(default_cfg['policy_model'],
                           dtype=default_cfg.get('dtype','bfloat16'),
                           gradient_checkpointing=False, device_map=DEVICE)
    ppo_ref = apply_lora(ppo_ref, r=default_cfg['lora_r'], alpha=default_cfg['lora_alpha'],
                          dropout=0.0, target_modules=default_cfg['lora_target_modules'])
    if Path(SFT_CKPT).exists():
        ppo_ref.load_adapter(SFT_CKPT, adapter_name='default')
    freeze_model(ppo_ref)

    vm = ValueModel(default_cfg['value_model'], freeze_backbone=True, use_8bit=False)

    if rm is None:
        print('ERROR: RM not loaded — cannot run PPO. Load RM first.')
    else:
        from transformers import AutoTokenizer
        rm_tok = AutoTokenizer.from_pretrained(default_cfg['reward_model'])
        rm_tok.padding_side = 'right'
        if rm_tok.pad_token is None: rm_tok.pad_token = rm_tok.eos_token

        pol_opt = torch.optim.AdamW(
            [p for p in ppo_policy.parameters() if p.requires_grad], lr=ppo_cfg.get('lr_policy', 1e-5))
        val_opt = torch.optim.AdamW(
            [p for p in vm.parameters() if p.requires_grad], lr=ppo_cfg.get('lr_value', 1e-4))

        all_prompts = [d['prompt'] for d in hh_train]
        mlog_ppo = MetricsLogger('runs/ppo/metrics.jsonl')
        ensure_dir('runs/ppo')

        for step in tqdm(range(ppo_cfg.get('update_steps', 200)), desc='PPO'):
            batch_prompts = random.sample(all_prompts, ppo_cfg.get('prompts_per_step', 8))
            clear_cache()
            rollout = ppo_rollout(
                policy_model=ppo_policy, ref_model=ppo_ref,
                value_model=vm, reward_model=rm, rm_tokenizer=rm_tok,
                policy_tokenizer=policy_tok, prompts=batch_prompts,
                max_new_tokens=ppo_cfg.get('max_new_tokens', 128),
                temperature=ppo_cfg.get('temperature', 0.7),
                top_p=ppo_cfg.get('top_p', 0.9),
                beta=ppo_cfg.get('beta', 0.1),
                device=DEVICE,
            )
            info = ppo_update(
                ppo_policy, vm, rollout, pol_opt, val_opt,
                ppo_epochs=ppo_cfg.get('ppo_epochs', 4),
                epsilon=ppo_cfg.get('epsilon', 0.2),
                device=DEVICE,
            )
            info['mean_reward'] = sum(rollout.task_rewards) / len(rollout.task_rewards)
            if (step + 1) % ppo_cfg.get('log_every', 5) == 0:
                mlog_ppo.log(step + 1, info)
                print(f'[PPO {step+1}] reward={info["mean_reward"]:.3f}  '
                      f'kl={info["kl_approx"]:.4f}  '
                      f'pol_loss={info["policy_loss"]:.4f}')

        save_peft_checkpoint(ppo_policy, 'runs/ppo/adapter')
        mlog_ppo.close()
        print('PPO complete.')
        print_memory()
else:
    print('RUN_PPO=False — skipping.')


## 14 · GRPO Training (C5)

Critic-free online RL. Requires RM for scoring.
Log degenerate batch fraction — if >30%, reward model discriminates poorly.

In [ ]:
if RUN_GRPO:
    import random
    from tqdm.notebook import tqdm

    if rm is None:
        print('ERROR: RM not loaded — cannot run GRPO.')
    else:
        clear_cache()
        from transformers import AutoTokenizer
        rm_tok_g = AutoTokenizer.from_pretrained(default_cfg['reward_model'])
        rm_tok_g.padding_side = 'right'
        if rm_tok_g.pad_token is None: rm_tok_g.pad_token = rm_tok_g.eos_token

        grpo_policy = load_policy(default_cfg['policy_model'],
                                   dtype=default_cfg.get('dtype','bfloat16'),
                                   gradient_checkpointing=True, device_map=DEVICE)
        grpo_policy = apply_lora(grpo_policy,
            r=default_cfg['lora_r'], alpha=default_cfg['lora_alpha'],
            dropout=default_cfg['lora_dropout'],
            target_modules=default_cfg['lora_target_modules'])
        if Path(SFT_CKPT).exists():
            grpo_policy.load_adapter(SFT_CKPT, adapter_name='default')

        grpo_ref = load_policy(default_cfg['policy_model'],
                                dtype=default_cfg.get('dtype','bfloat16'),
                                gradient_checkpointing=False, device_map=DEVICE)
        grpo_ref = apply_lora(grpo_ref, r=default_cfg['lora_r'],
                               alpha=default_cfg['lora_alpha'], dropout=0.0,
                               target_modules=default_cfg['lora_target_modules'])
        if Path(SFT_CKPT).exists():
            grpo_ref.load_adapter(SFT_CKPT, adapter_name='default')
        freeze_model(grpo_ref)

        def grpo_reward_fn(texts):
            return score_texts(rm, rm_tok_g, texts)

        g_opt = torch.optim.AdamW(
            [p for p in grpo_policy.parameters() if p.requires_grad],
            lr=grpo_cfg.get('lr', 1e-5))
        all_prompts = [d['prompt'] for d in hh_train]
        mlog_grpo = MetricsLogger('runs/grpo/metrics.jsonl')
        ensure_dir('runs/grpo')

        for step in tqdm(range(grpo_cfg.get('update_steps', 200)), desc='GRPO'):
            batch_prompts = random.sample(all_prompts, grpo_cfg.get('prompts_per_step', 8))
            clear_cache()
            rollout = grpo_rollout(
                policy_model=grpo_policy, ref_model=grpo_ref,
                reward_fn=grpo_reward_fn, policy_tokenizer=policy_tok,
                prompts=batch_prompts, K=grpo_cfg.get('K', 4),
                max_new_tokens=grpo_cfg.get('max_new_tokens', 128),
                temperature=grpo_cfg.get('temperature', 0.7),
                device=DEVICE,
            )
            info = grpo_update(
                grpo_policy, grpo_ref, rollout, g_opt,
                epsilon=grpo_cfg.get('epsilon', 0.2),
                beta=grpo_cfg.get('beta', 0.1),
                device=DEVICE,
            )
            if (step + 1) % grpo_cfg.get('log_every', 5) == 0:
                mlog_grpo.log(step + 1, info)
                print(f'[GRPO {step+1}] reward={info["mean_reward"]:.3f}  '
                      f'degen={info["degenerate"]:.2f}  '
                      f'kl={info["kl_loss"]:.4f}')

        save_peft_checkpoint(grpo_policy, 'runs/grpo/adapter')
        mlog_grpo.close()
        print('GRPO complete.')
        print_memory()
else:
    print('RUN_GRPO=False — skipping.')


## 15 · RLVR — RL with Verifiable Rewards (C6)

GRPO on GSM8K with binary `r_v ∈ {0,1}` instead of a learned RM.
Initialises from SFT checkpoint. No RM needed.

In [ ]:
if RUN_RLVR:
    import random
    from tqdm.notebook import tqdm

    clear_cache()
    rlvr_policy = load_policy(default_cfg['policy_model'],
                               dtype=default_cfg.get('dtype','bfloat16'),
                               gradient_checkpointing=True, device_map=DEVICE)
    rlvr_policy = apply_lora(rlvr_policy,
        r=default_cfg['lora_r'], alpha=default_cfg['lora_alpha'],
        dropout=default_cfg['lora_dropout'],
        target_modules=default_cfg['lora_target_modules'])
    if Path(SFT_CKPT).exists():
        rlvr_policy.load_adapter(SFT_CKPT, adapter_name='default')
        print('Loaded SFT adapter for RLVR policy.')

    rlvr_ref = load_policy(default_cfg['policy_model'],
                            dtype=default_cfg.get('dtype','bfloat16'),
                            gradient_checkpointing=False, device_map=DEVICE)
    rlvr_ref = apply_lora(rlvr_ref, r=default_cfg['lora_r'],
                           alpha=default_cfg['lora_alpha'], dropout=0.0,
                           target_modules=default_cfg['lora_target_modules'])
    if Path(SFT_CKPT).exists():
        rlvr_ref.load_adapter(SFT_CKPT, adapter_name='default')
    freeze_model(rlvr_ref)

    rv_opt = torch.optim.AdamW(
        [p for p in rlvr_policy.parameters() if p.requires_grad],
        lr=rlvr_cfg.get('lr', 1e-5))
    mlog_rlvr = MetricsLogger('runs/rlvr/metrics.jsonl')
    ensure_dir('runs/rlvr')

    for step in tqdm(range(rlvr_cfg.get('update_steps', 300)), desc='RLVR'):
        batch = random.sample(gsm_train, rlvr_cfg.get('prompts_per_step', 8))
        prompts_b = [it['prompt'] for it in batch]
        golds_b   = [it['gold_answer'] for it in batch]
        clear_cache()
        rollout = rlvr_rollout(
            policy_model=rlvr_policy, ref_model=rlvr_ref,
            policy_tokenizer=policy_tok, prompts=prompts_b,
            gold_answers=golds_b, K=rlvr_cfg.get('K', 4),
            max_new_tokens=rlvr_cfg.get('max_new_tokens', 256),
            temperature=rlvr_cfg.get('temperature', 0.7),
            device=DEVICE,
        )
        from alignment.grpo import grpo_update
        info = grpo_update(
            rlvr_policy, rlvr_ref, rollout, rv_opt,
            epsilon=rlvr_cfg.get('epsilon', 0.2),
            beta=rlvr_cfg.get('beta', 0.05),
            device=DEVICE,
        )
        if (step + 1) % rlvr_cfg.get('log_every', 5) == 0:
            mlog_rlvr.log(step + 1, info)

        if (step + 1) % rlvr_cfg.get('eval_every', 25) == 0:
            p1 = eval_gsm8k_pass_at_1(
                rlvr_policy, policy_tok, gsm_test,
                max_new_tokens=rlvr_cfg.get('max_new_tokens', 256),
                device=DEVICE, max_samples=100,
            )
            print(f'[RLVR {step+1}] pass@1={p1:.4f}  reward={info["mean_reward"]:.3f}')
            mlog_rlvr.log(step + 1, {'pass_at_1': p1})

    save_peft_checkpoint(rlvr_policy, 'runs/rlvr/adapter')
    mlog_rlvr.close()
    print('RLVR complete.')
    print_memory()
else:
    print('RUN_RLVR=False — skipping.')


## 16 · Evaluation (C8)

Win-rate vs SFT baseline, KL from πref, and sample response table.

In [ ]:
if RUN_EVAL:
    from transformers import AutoTokenizer

    eval_prompts = [d['prompt'] for d in hh_test[:200]]

    if rm is None:
        print('Cannot run win-rate eval without RM. Load RM first.')
    else:
        rm_tok_e = AutoTokenizer.from_pretrained(default_cfg['reward_model'])
        rm_tok_e.padding_side = 'right'
        if rm_tok_e.pad_token is None: rm_tok_e.pad_token = rm_tok_e.eos_token

        def score_batch(model, prompts, batch_size=8):
            all_scores = []
            for i in range(0, len(prompts), batch_size):
                bp = prompts[i:i+batch_size]
                resps = generate_responses(model, policy_tok, bp,
                                           max_new_tokens=128, do_sample=False, device=DEVICE)
                texts = [p + ' ' + r for p, r in zip(bp, resps)]
                all_scores.extend(score_texts(rm, rm_tok_e, texts))
            return all_scores

        print('Scoring SFT baseline ...')
        sft_eval_model = load_policy(default_cfg['policy_model'],
                                      dtype=default_cfg.get('dtype','bfloat16'),
                                      gradient_checkpointing=False, device_map=DEVICE)
        sft_eval_model = apply_lora(sft_eval_model, r=default_cfg['lora_r'],
                                     alpha=default_cfg['lora_alpha'], dropout=0.0,
                                     target_modules=default_cfg['lora_target_modules'])
        if Path(SFT_CKPT).exists():
            sft_eval_model.load_adapter(SFT_CKPT, adapter_name='default')
        sft_scores = score_batch(sft_eval_model, eval_prompts)
        print(f'SFT mean score: {sum(sft_scores)/len(sft_scores):.4f}')

        eval_results = {'sft': {'mean_score': sum(sft_scores)/len(sft_scores)}}
        for method in ['dpo', 'ppo', 'grpo']:
            ap = f'runs/{method}/adapter'
            if not Path(ap).exists():
                print(f'  {method}: adapter not found, skipping')
                continue
            print(f'  Scoring {method} ...')
            m = load_policy(default_cfg['policy_model'],
                             dtype=default_cfg.get('dtype','bfloat16'),
                             gradient_checkpointing=False, device_map=DEVICE)
            m = apply_lora(m, r=default_cfg['lora_r'], alpha=default_cfg['lora_alpha'],
                            dropout=0.0, target_modules=default_cfg['lora_target_modules'])
            m.load_adapter(ap, adapter_name='default')
            scores = score_batch(m, eval_prompts)
            wr = win_rate(scores, sft_scores)
            eval_results[method] = {'win_rate': wr, 'mean_score': sum(scores)/len(scores)}
            print(f'  {method}: win_rate={wr:.4f}  mean_score={sum(scores)/len(scores):.4f}')
            del m; clear_cache()

        print('\n=== Evaluation Summary ===')
        for method, res in eval_results.items():
            print(f'  {method}: {res}')
else:
    print('RUN_EVAL=False — skipping.')


## 17 · Training Curves

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

for method in ['sft', 'rm', 'dpo', 'ppo', 'grpo', 'rlvr']:
    log_path = Path(f'runs/{method}/metrics.jsonl')
    if log_path.exists():
        plot_training_curves(log_path, save_path=f'runs/{method}/curves.png')
        print(f'Saved runs/{method}/curves.png')
    else:
        print(f'No log found for {method} — skipping')


## 18 · Sample Generations Table

In [ ]:
sample_prompts = [d['prompt'] for d in hh_test[:5]]
print('=== Sample Generations (5 prompts) ===')
for i, prompt in enumerate(sample_prompts):
    print(f'\n--- Prompt {i+1} ---')
    print(prompt[-200:])
    # SFT response
    try:
        resp = generate_responses(sft_eval_model, policy_tok, [prompt],
                                  max_new_tokens=100, do_sample=False, device=DEVICE)[0]
        print(f'  [SFT]: {resp[:150]}')
    except Exception:
        print('  [SFT]: (model not loaded)')


## 19 · Save Outputs

In [ ]:
import json
from pathlib import Path

ensure_dir('runs')
# Write a summary JSON of all results
summary = {
    'methods_run': [m for m, flag in [('sft', RUN_SFT), ('rm', RUN_RM),
                                       ('dpo', RUN_DPO), ('ppo', RUN_PPO),
                                       ('grpo', RUN_GRPO), ('rlvr', RUN_RLVR)] if flag],
}
if 'eval_results' in dir():
    summary['eval'] = eval_results

Path('runs/summary.json').write_text(json.dumps(summary, indent=2))
print('Saved runs/summary.json')
print(json.dumps(summary, indent=2))


## 20 · Package Source Files (scripts.zip)

In [ ]:
!python tools/make_scripts_zip.py --out scripts.zip
import os
size_kb = os.path.getsize('scripts.zip') / 1024
print(f'scripts.zip created ({size_kb:.1f} KB)')


## 21 · Final Summary

All methods have been trained and evaluated. Key outputs:

| Path | Contents |
|---|---|
| `runs/sft/sft_adapter/` | SFT LoRA adapters (πref + π⁰_θ) |
| `runs/rm/rm.pt` | Trained reward model checkpoint |
| `runs/dpo/adapter/` | DPO LoRA adapters |
| `runs/ppo/adapter/` | PPO LoRA adapters |
| `runs/grpo/adapter/` | GRPO LoRA adapters |
| `runs/rlvr/adapter/` | RLVR LoRA adapters |
| `runs/*/metrics.jsonl` | Per-step training metrics |
| `runs/summary.json` | Evaluation results summary |
| `scripts.zip` | All source code packaged |

> Check `runs/eval/sample_table.md` for the side-by-side response comparison.